# N062 · 递归函数契约与调用栈

**目标：** 按子问题含义编写递归而非猜测执行顺序。

**先修：** N004, N011, N013。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 基例；规模递减；参数与返回值；栈空间；尾递归不自动优化。

**配套讲解来源：** [同名逐章意见](../../comment/062_recursion_contracts.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章不学新算法，而是学"怎么把递归写对"。很多人写递归时总想追踪"程序先调谁、后调谁、返回到哪一行"，越追越晕。正确的心法是：**先给函数写一份"契约"（这个函数接收什么、保证返回什么），再相信契约、调用它自己**，执行顺序交给 Python。

具体到数字的小例子：

- `factorial(5)` 应返回 120。契约是"`factorial(n)` 返回前 n 个正整数的乘积"。于是 `factorial(5) = 5 × factorial(4)`，因为 factorial(4) 按契约"就是" 24，所以 5×24=120。你不需要在脑子里展开 5 层调用。
- `fib_recursive(6)` 应返回 8。斐波那契契约：`fib(0)=0, fib(1)=1, fib(n)=fib(n-1)+fib(n-2)`。
- `trace_calls(5)` 不算数，而是把整个调用过程录下来给你看：它返回 15 条 `(参数, 深度)` 记录，正好让我们亲眼数一数"fib(5) 到底把 fib(1) 算了几遍"——答案是 5 遍。这种重复正是下一章记忆化的动机。

本章还要搞懂三件配套的事：每条递归路径必须越走越接近"基本情形"（否则无限递归）；每层调用都要占一块"栈"内存；以及一个反直觉的事实——Python 不会把"尾巴上的递归调用"自动优化成循环，栈该多深还是多深。

## 2. 基础知识：先读懂这些词

- **函数契约**：一句话写清"输入什么含义的参数、保证返回什么含义的结果"。写递归时契约就是你的靠山：调用 `factorial(n-1)` 时不必关心它内部怎么跑，只需要相信它按契约返回正确答案。
- **基本情形（基例）**：递归停止的条件，是最小、最简单、能直接给答案的输入。`n==0` 返回 1（阶乘）、`n<2` 返回 n（斐波那契）都是基例。没有基例的递归就是永动机。
- **规模递减（参数严格下降）**：每次递归调用的参数必须比上一层"更小"（本章是 n-1、n-2，严格减且不小于 0），这样任何调用链走有限步后必然撞进基例，程序才会停。
- **调用栈（栈空间）**：Python 每调用一层函数就在栈上留一块"工作记录"（局部变量、回到哪里继续）。递归 n 层就有 n 块。`factorial(10⁵)` 理论上乘法不算慢，但栈要叠十万层，会先撞上 Python 默认约 1000 层的递归上限——这就是"栈空间也是一种成本"的直观含义。
- **调用树**：把每次调用画成一个节点、它的递归调用画成孩子，得到一棵树。阶乘的调用树是一条链（每层只调一次自己）；斐波那契是一棵二叉树，且树里塞满了"参数相同"的重复节点。
- **重复子问题**：同一个参数被不同路径反复计算。fib(5) 的树里 fib(1) 出现 5 次、fib(2) 出现 3 次——每次都从零算起。这是朴素递归指数级爆炸的根源，也是后续"记忆化（memoization：把算过的答案存起来，下次直接查，不再重算）"要治的病。
- **尾递归不自动优化**：如果函数最后一步"只返回递归调用本身"叫尾递归，有些语言会把它改写成循环省掉栈。Python 明确不做这件事，所以指望"反正尾递归不会爆栈"在 Python 里行不通。

## 3. 思路推导：从小例子开始

Step 1：写契约。`factorial(n)`：n 是非负整数，返回 1×2×…×n（n=0 时返回 1）。`fib_recursive(n)`：返回斐波那契第 n 项（第 0 项 0、第 1 项 1）。`trace_calls(n)`：返回朴素 fib(n) 全部调用的 `(参数, 深度)` 列表。

Step 2：找基例。阶乘的基例是 n==0（乘法空积为 1）；fib 的基例是 n<2（0 和 1 直接给出）。

Step 3：写递归步，并且只"信任契约"地写：`factorial(n) = n * factorial(n-1)`——假设 n-1 的答案是对的，乘上 n 就是对的；`fib(n) = fib(n-1) + fib(n-2)` 同理。

Step 4：挡非法输入。三个函数开头都用 `if n<0: raise ValueError` 挡负数：负数在契约里不存在，与其无限递归到爆栈，不如立刻报错。

Step 5：用 `trace_calls` 把执行过程"录像"，亲眼看见重复。

手算演示（notebook 小实例原数据）：`trace_calls(5)` 录下的 15 条记录按发生顺序是：

| 调用序号 | 参数 | 调用深度 |
| --- | --- | --- |
| 1 | 5 | 0 |
| 2 | 4 | 1 |
| 3 | 3 | 2 |
| 4 | 2 | 3 |
| 5 | 1 | 4 |
| 6 | 0 | 5 |
| 7 | 1 | 4 |
| 8 | 2 | 3 |
| 9 | 1 | 4 |
| 10 | 0 | 5 |
| 11 | 3 | 2 |
| 12 | 2 | 3 |
| 13 | 1 | 4 |
| 14 | 0 | 5 |
| 15 | 1 | 4 |

怎么读这张表？前 6 行是"最左边一条链"：5→4→3→2→1→0 一路扎到底（深度 0 到 5）；第 7 行是 fib(2) 的右孩子 fib(1)；第 8 行开始回溯到 fib(3) 的右子树……notebook 的第二张表把这 15 条按参数去重计数：参数 5 出现 1 次、4 出现 1 次、3 出现 2 次、2 出现 3 次、1 出现 5 次、0 出现 3 次。fib(1) 算了 5 遍、fib(2) 算了 3 遍——重复子问题当场现形。这张表与 notebook 小实例生成的两张表一致。

## 4. 核心代码：factorial

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def factorial(n):
    if n < 0:
        raise ValueError('n must be nonnegative')
    return 1 if n == 0 else n * factorial(n - 1)
```

### 逐段读懂代码

### 1. `factorial`：一条链的递归

```python
def factorial(n):
    if n < 0:
        raise ValueError('n must be nonnegative')
    return 1 if n == 0 else n * factorial(n - 1)
```

- 第一行是守卫：负数不在契约范围内，主动报 ValueError 而不是递归到负无穷。
- 最后一行是一个三元表达式，同时写完"基例 + 递归步"：n 到 0 返回 1；否则返回 `n*factorial(n-1)`。注意乘法发生在"内层调用返回之后"，也就是说 factorial(5) 要等 factorial(4) 回来才能乘——这正是它占用栈的原因。
- 参数每层严格减 1，所以任何合法输入走有限步必停在 0。

### 2. `fib_recursive`：一棵树的递归

```python
def fib_recursive(n):
    if n < 0:
        raise ValueError('n must be nonnegative')
    return n if n < 2 else fib_recursive(n - 1) + fib_recursive(n - 2)
```

- 结构与阶乘同款：守卫 + 三元表达式。
- 基例合并成 `n<2`：n 是 0 就返回 0、是 1 就返回 1，`return n` 一箭双雕。
- 递归步一次分出两个调用，调用树从此变成二叉树。代价是恐怖的重复：每个 fib(k) 会被上层各种路径反复引爆，调用总次数按公式是 `2*fib(n+1)-1`（notebook 的测试正是用这条公式验收的）。
- notebook 特意叮嘱：别拿很大的 n 去跑它，教学内核会被这棵指数级大树卡死——比如 n=40 就有约 3.3 亿次调用。

### 3. `trace_calls`：给调用过程录像

```python
def trace_calls(n):
    if n < 0:
        raise ValueError('n must be nonnegative')
    calls = []

    def visit(k, depth):
        calls.append((k, depth))
        if k < 2:
            return k
        return visit(k - 1, depth + 1) + visit(k - 2, depth + 1)
    visit(n, 0)
    return calls
```

- 外层守卫照旧；`calls` 列表是"录像带"；`visit` 是内部函数，比 fib 多带一个 `depth` 参数记录当前深度（根为 0）。
- `calls.append((k,depth))` 放在函数第一行：进入即记录，这样列表顺序就是真实的发生顺序。
- `if k<2: return k` 与 fib 的基例一模一样；递归步把 depth+1 传给孩子，深度信息随调用下传。
- 计算逻辑与 `fib_recursive` 完全一致（同样的重复模式），所以这盘录像带忠实还原了朴素 fib 的行为：它记录的次数公式同样是 `2*fib(n+1)-1`，第三节手算的 trace_calls(5) 共 15 条 = 2×fib(6)-1 = 2×8-1。

## 5. 分段实现：按顺序运行

**本章接口：** `factorial(n)`、`fib_recursive(n)`、`trace_calls(n)`

### factorial

In [1]:
def factorial(n):
    if n < 0:
        raise ValueError('n must be nonnegative')
    return 1 if n == 0 else n * factorial(n - 1)

### fib_recursive

In [2]:
def fib_recursive(n):
    if n < 0:
        raise ValueError('n must be nonnegative')
    return n if n < 2 else fib_recursive(n - 1) + fib_recursive(n - 2)

### trace_calls

In [3]:
def trace_calls(n):
    if n < 0:
        raise ValueError('n must be nonnegative')
    calls = []

    def visit(k, depth):
        calls.append((k, depth))
        if k < 2:
            return k
        return visit(k - 1, depth + 1) + visit(k - 2, depth + 1)
    visit(n, 0)
    return calls

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

from collections import Counter
calls=trace_calls(5)
show_table(['调用序号','参数','调用深度'],[(i,k,d) for i,(k,d) in enumerate(calls)])
show_table(['子问题参数','重复次数'],sorted(Counter(k for k,_ in calls).items()))

调用序号,参数,调用深度
0,5,0
1,4,1
2,3,2
3,2,3
4,1,4
5,0,4
6,1,3
7,2,2
8,1,3
9,0,3


子问题参数,重复次数
0,3
1,5
2,3
3,2
4,1
5,1


## 7. 正确性、适用条件与复杂度

在基本情形正确的前提下，假设更小参数的调用正确，由递推定义立即得到当前结果。参数严格下降保证有限非负n终止。

**代价：** 阶乘O(n)次乘法、O(n)栈深；朴素fib O(φⁿ)时间/O(n)栈深；大整数乘法另有位代价。

### 展开理解

为什么这样写就一定对？我们对 n 从小到大信任过去：n=0 时函数直接给 1，显然对；假设"所有比 n 小的参数都返回正确答案"，那么 `n * factorial(n-1)` 就是 n 乘上 1×2×…×(n-1)，正是前 n 个正整数的乘积——所以 n 也对。这份"从小推到大"的信任链没有断点，结论对所有非负 n 成立。fib 同理：只要 fib(n-1)、fib(n-2) 对，相加就是第 n 项。

为什么一定会停？因为每次调用参数都严格变小（n-1、n-2），而参数又被守卫钉死在 0 以上，所以任何一条调用链的参数是严格递减的非负整数序列，走有限步必然小于 2 落进基例。没有基例、或参数不减小（比如写成 `fib(n)+fib(n-2)`），这条论证立刻塌掉，程序就真的停不下来。

复杂度要分"时间"和"栈"两笔账。阶乘：n 层调用每层一次乘法，时间 O(n)、栈深 O(n)——n=10⁵ 就是十万层，先把 Python 默认约 1000 层的递归上限撞爆，这提醒我们"栈也是资源"。斐波那契：调用次数 2×fib(n+1)-1，而 fib(n) 以黄金比 φ≈1.618 的速度膨胀，所以时间约 O(1.618ⁿ)。具体感受一下指数的威力：n=30 时约 270 万次调用（一眨眼），n=40 时约 3.3 亿次（明显卡），n=50 时约 400 亿次（别想了）。但它左倾的调用树最深只走 n 层（一路 fib(n-1) 扎到底），所以栈深只有 O(n)。`trace_calls` 复杂度与 fib 相同，因此 notebook 提醒别用大 n 调它。

## 8. 单元测试：每个用例在检查什么

```python
assert factorial(0) == 1 and factorial(5) == 120
```
正常值加边界值：0 是阶乘的最小合法输入（空积为 1，最容易被手滑写成 0），5 是随手可验的常规值。

```python
assert fib_recursive(0) == 0 and fib_recursive(1) == 1
```
边界用例：斐波那契两个基例本身必须原样返回 0 和 1，这是整棵信任链的地基。

```python
assert len(trace_calls(4)) == 9
```
正常用例：n=4 的调用次数应为 2×fib(5)-1=2×5-1=9 条。这条断言验证"录像带的长度"符合调用次数公式，侧面确认 trace 的计算模式与 fib 完全一致。

```python
for n in range(9):
    assert factorial(n) == math.factorial(n)
    assert len(trace_calls(n)) == 2 * fib_recursive(n + 1) - 1
```
系统对照：阶乘和标准库 `math.factorial` 在 n=0..8 逐一比对；trace 的长度和理论公式在 n=0..8 逐一比对。n=0 时 trace 只有 1 条（根就是基例），这条边界也被覆盖。

```python
for fn in [factorial, fib_recursive, trace_calls]:
    try:
        fn(-1)
    except ValueError:
        pass
    else:
        raise AssertionError('negative arguments must be rejected')
```
异常契约测试：三个函数喂 -1 都必须抛 ValueError。`try/except/else` 的结构值得学——except 里 `pass` 表示"抛了就合格"；else 分支只有"没抛异常"才会执行，此时主动 raise 宣告失败。

In [5]:
import math

assert factorial(0) == 1 and factorial(5) == 120

assert fib_recursive(0) == 0 and fib_recursive(1) == 1

assert len(trace_calls(4)) == 9

for n in range(9):
    assert factorial(n) == math.factorial(n)
    assert len(trace_calls(n)) == 2 * fib_recursive(n + 1) - 1

for fn in [factorial, fib_recursive, trace_calls]:
    try:
        fn(-1)
    except ValueError:
        pass
    else:
        raise AssertionError('negative arguments must be rejected')

print('N062: 所有本章断言通过')

N062: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 将一个递归函数改成显式栈。

**练习 2：** 构造缺失基例的错误并解释。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- 练习 1（把递归改成显式栈）：任选 factorial 或 fib，用 `list` 当栈模拟递归。提示：递归本质是"系统帮你管一个返回地址栈"，你自己管时，栈里存的应是"待处理的参数"（改循环版 factorial）或"待展开/待求和的节点"（改 fib 时可存 `(k,'expand')`、`(v,'add')` 两类记号）。边界是空栈即结束、初始压入 n。手算例子建议从 trace_calls(5) 的 15 条顺序反过来推压栈/弹栈过程。
- 练习 2（构造缺失基例的错误并解释）：把 `if n==0` 删掉或改成 `if n==1`，喂 0 进去。提示：观察 `RecursionError`（Python 对无限递归的保护）而不是无限卡死；解释时说清"参数序列 0,-1,-2,… 永远撞不到基例"；再演示"参数不减"的版本（比如调用 factorial(n)），说明另一种死法。写清输入（0 或任何数）、预期（异常）、边界（合法下界的正下方）。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def factorial(n):
    if n < 0:
        raise ValueError('n must be nonnegative')
    return 1 if n == 0 else n * factorial(n - 1)

def fib_recursive(n):
    if n < 0:
        raise ValueError('n must be nonnegative')
    return n if n < 2 else fib_recursive(n - 1) + fib_recursive(n - 2)

def trace_calls(n):
    if n < 0:
        raise ValueError('n must be nonnegative')
    calls = []

    def visit(k, depth):
        calls.append((k, depth))
        if k < 2:
            return k
        return visit(k - 1, depth + 1) + visit(k - 2, depth + 1)
    visit(n, 0)
    return calls

# 示例与回归测试
import math

assert factorial(0) == 1 and factorial(5) == 120

assert fib_recursive(0) == 0 and fib_recursive(1) == 1

assert len(trace_calls(4)) == 9

for n in range(9):
    assert factorial(n) == math.factorial(n)
    assert len(trace_calls(n)) == 2 * fib_recursive(n + 1) - 1

for fn in [factorial, fib_recursive, trace_calls]:
    try:
        fn(-1)
    except ValueError:
        pass
    else:
        raise AssertionError('negative arguments must be rejected')

print('N062: 所有本章断言通过')

N062: 所有本章断言通过


## 11. 代表题与迁移

- **509. Fibonacci Number（斐波那契数）**：`fib_recursive` 的原题，练的是"契约 + 基例 + 递归步"三件套，也顺便暴露朴素递归的重复子问题。
- **50. Pow(x, n)（Pow(x, n)）**：下一章 N063 的主角，在这里作为预告——它同样用递归契约思考（"power(x,n) 返回 x 的 n 次方"），但递归步会把 n 减半，复杂度直接从线性掉到对数。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。